## Evaluating Fine-Tuned Model

In [10]:
import json

# Load predictions
with open('data/outputs/finetuned_model_predictions.json', 'r') as f:
    predictions = json.load(f)

def evaluate_prediction(pred):
    gt = pred["ground_truth"]
    out = pred["parsed_output"]
    score = 0

    # Department (0.3)
    if out.get("department") == gt.get("department"):
        score += 0.3

    # Symptoms (0.3)
    gt_symptoms = [s.lower() for s in gt.get("symptoms", [])]
    out_symptoms = [s.lower() for s in out.get("symptoms", [])]
    if gt_symptoms:
        matches = sum(1 for s in gt_symptoms if s in out_symptoms)
        if matches == len(gt_symptoms):
            score += 0.3
        else:
            score += (matches / len(gt_symptoms)) * 0.3

    # Sentiment (0.2)
    if out.get("sentiment") == gt.get("sentiment"):
        score += 0.2

    # Urgency level (0.2)
    if out.get("urgency_level") == gt.get("urgency_level"):
        score += 0.2

    return round(score, 4)

# Evaluate all predictions
results = []
for pred in predictions:
    s = evaluate_prediction(pred)
    results.append({
        "id": pred["id"],
        "score": s,
        "valid_json": pred.get("valid_json", False)
    })

scores = [r["score"] for r in results]
avg_score = sum(scores) / len(scores)

print(f"Total predictions: {len(scores)}")
print(f"Average score: {avg_score:.4f}")
print(f"Min score: {min(scores):.4f}")
print(f"Max score: {max(scores):.4f}")
print(f"\nPer-prediction scores:")
for r in results:
    print(f"  ID {r['id']}: {r['score']:.4f}  (valid JSON: {r['valid_json']})")

Total predictions: 760
Average score: 0.7320
Min score: 0.0000
Max score: 1.0000

Per-prediction scores:
  ID 0: 1.0000  (valid JSON: True)
  ID 1: 0.6500  (valid JSON: True)
  ID 2: 0.8000  (valid JSON: True)
  ID 3: 0.0000  (valid JSON: True)
  ID 4: 0.7000  (valid JSON: True)
  ID 5: 0.4000  (valid JSON: True)
  ID 6: 0.5000  (valid JSON: True)
  ID 7: 0.7000  (valid JSON: True)
  ID 8: 0.7000  (valid JSON: True)
  ID 9: 0.7000  (valid JSON: True)
  ID 10: 1.0000  (valid JSON: True)
  ID 11: 0.6000  (valid JSON: True)
  ID 12: 0.5000  (valid JSON: True)
  ID 13: 1.0000  (valid JSON: True)
  ID 14: 1.0000  (valid JSON: True)
  ID 15: 0.7000  (valid JSON: True)
  ID 16: 1.0000  (valid JSON: True)
  ID 17: 0.6500  (valid JSON: True)
  ID 18: 0.8500  (valid JSON: True)
  ID 19: 0.7000  (valid JSON: True)
  ID 20: 0.5000  (valid JSON: True)
  ID 21: 0.8000  (valid JSON: True)
  ID 22: 1.0000  (valid JSON: True)
  ID 23: 0.6500  (valid JSON: True)
  ID 24: 1.0000  (valid JSON: True)
  ID 

## Evaluating Base Model (added explanations for some so removing that first before evaluating)

In [26]:
import json
import re

# Load base model predictions
with open('data/outputs/base_model_predictions.json', 'r') as f:
    base_predictions = json.load(f)

# Normalization
URGENCY_NORMALIZE = {
    "low": "Low", "medium": "Medium", "moderate": "Medium",
    "high": "High", "critical": "Critical", "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(str(value).lower().strip(), value)

def parse_output(raw):
    if not raw:
        return None
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

# Check nulls before re-parsing
null_before = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Null entries BEFORE re-parsing: {null_before}/{len(base_predictions)}")

# Re-parse null entries
for p in base_predictions:
    if p["parsed_output"] is None:
        p["parsed_output"] = parse_output(p["raw_model_output"])

# Check nulls after re-parsing
null_after = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Null entries AFTER re-parsing:  {null_after}/{len(base_predictions)}")
print(f"Recovered by re-parsing:        {null_before - null_after}")

def evaluate_prediction(pred):
    gt = pred["ground_truth"]
    out = pred["parsed_output"]

    if out is None:
        return 0.0

    score = 0

    # Department (0.3)
    if out.get("department") == gt.get("department"):
        score += 0.3

    # Symptoms (0.3)
    gt_symptoms = [s.lower() for s in gt.get("symptoms", [])]
    out_symptoms = [s.lower() for s in out.get("symptoms", [])]
    if gt_symptoms:
        matches = sum(1 for s in gt_symptoms if s in out_symptoms)
        if matches == len(gt_symptoms):
            score += 0.3
        else:
            score += (matches / len(gt_symptoms)) * 0.3

    # Sentiment (0.2)
    if out.get("sentiment") == gt.get("sentiment"):
        score += 0.2

    # Urgency level (0.2) — normalized
    if normalize_urgency(out.get("urgency_level")) == normalize_urgency(gt.get("urgency_level")):
        score += 0.2

    return round(score, 4)

# Evaluate all predictions
base_results = []
for pred in base_predictions:
    s = evaluate_prediction(pred)
    base_results.append({
        "id": pred["id"],
        "score": s,
        "valid_json": pred.get("valid_json", False)
    })

base_scores = [r["score"] for r in base_results]
base_avg = sum(base_scores) / len(base_scores)

print(f"\nTotal predictions: {len(base_scores)}")
print(f"Average score:     {base_avg:.4f}")
print(f"Min score:         {min(base_scores):.4f}")
print(f"Max score:         {max(base_scores):.4f}")
print(f"\nPer-prediction scores:")
for r in base_results:
    print(f"  ID {r['id']}: {r['score']:.4f}  (valid JSON: {r['valid_json']})")

Null entries BEFORE re-parsing: 1/760
Null entries AFTER re-parsing:  0/760
Recovered by re-parsing:        1

Total predictions: 760
Average score:     0.4246
Min score:         0.0000
Max score:         1.0000

Per-prediction scores:
  ID 0: 0.5000  (valid JSON: True)
  ID 1: 0.4500  (valid JSON: True)
  ID 2: 0.7000  (valid JSON: True)
  ID 3: 0.3000  (valid JSON: True)
  ID 4: 0.6500  (valid JSON: True)
  ID 5: 0.2000  (valid JSON: True)
  ID 6: 0.2000  (valid JSON: True)
  ID 7: 0.5000  (valid JSON: True)
  ID 8: 0.5000  (valid JSON: True)
  ID 9: 0.0000  (valid JSON: True)
  ID 10: 0.4000  (valid JSON: True)
  ID 11: 0.7000  (valid JSON: True)
  ID 12: 0.0000  (valid JSON: True)
  ID 13: 0.5000  (valid JSON: True)
  ID 14: 1.0000  (valid JSON: True)
  ID 15: 0.5000  (valid JSON: True)
  ID 16: 1.0000  (valid JSON: True)
  ID 17: 0.6500  (valid JSON: True)
  ID 18: 0.6500  (valid JSON: True)
  ID 19: 0.2000  (valid JSON: True)
  ID 20: 0.2000  (valid JSON: True)
  ID 21: 0.8000  (

## Specifically Urgency Level

In [30]:
gt_urgencies = set(p["ground_truth"].get("urgency_level") for p in base_predictions)
out_urgencies = set(p["parsed_output"].get("urgency_level") for p in base_predictions if p["parsed_output"] is not None)

print("Ground truth unique urgency levels:", gt_urgencies)
print("Model output unique urgency levels:", out_urgencies)

Ground truth unique urgency levels: {'Low', 'Medium', 'High'}
Model output unique urgency levels: {'high', 'moderate', 'Non-Emergency', 'Medium', 'Moderate', 'Low', 'Non-emergency', 'low', 'High'}


In [31]:
ft_urgencies = set(p["parsed_output"].get("urgency_level") for p in predictions if p["parsed_output"] is not None)
gt_urgencies = set(p["ground_truth"].get("urgency_level") for p in predictions)

print("Ground truth unique urgency levels:", gt_urgencies)
print("Fine-tuned model unique urgency levels:", ft_urgencies)

Ground truth unique urgency levels: {'Low', 'Medium', 'High'}
Fine-tuned model unique urgency levels: {'Low', 'Medium', 'High'}


In [32]:
URGENCY_NORMALIZE = {
    "low": "Low",
    "medium": "Medium",
    "moderate": "Medium",
    "high": "High",
    "critical": "Critical",
    "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(value.lower().strip(), value)

URGENCY_ORDER = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}

def urgency_analysis(preds, model_name):
    lower, higher, wrong_other = 0, 0, 0

    for pred in preds:
        gt = pred["ground_truth"]
        out = pred["parsed_output"]

        if out is None:
            continue

        gt_urgency = normalize_urgency(gt.get("urgency_level"))
        out_urgency = normalize_urgency(out.get("urgency_level"))

        # Skip correct predictions
        if gt_urgency == out_urgency:
            continue

        gt_rank = URGENCY_ORDER.get(gt_urgency)
        out_rank = URGENCY_ORDER.get(out_urgency)

        if gt_rank is None or out_rank is None:
            wrong_other += 1
        elif out_rank < gt_rank:
            lower += 1
        elif out_rank > gt_rank:
            higher += 1

    total_wrong = lower + higher + wrong_other
    print(f" {model_name} ")
    print(f"Total wrong urgency predictions: {total_wrong}")
    print(f"  Predicted LOWER than ground truth:  {lower}")
    print(f"  Predicted HIGHER than ground truth: {higher}")
    if wrong_other:
        print(f"  Unexpected/unrecognised values:     {wrong_other}")
    print()

urgency_analysis(base_predictions, "Base Model")
urgency_analysis(predictions, "Fine-tuned Model")

 Base Model 
Total wrong urgency predictions: 270
  Predicted LOWER than ground truth:  7
  Predicted HIGHER than ground truth: 263

 Fine-tuned Model 
Total wrong urgency predictions: 67
  Predicted LOWER than ground truth:  36
  Predicted HIGHER than ground truth: 31



## Summary of all experiments (with added normalization for urgency)

In [33]:
import json
import re

# Load predictions
with open('data/outputs/finetuned_model_predictions.json', 'r') as f:
    predictions = json.load(f)

with open('data/outputs/base_model_predictions.json', 'r') as f:
    base_predictions = json.load(f)

# Normalization
URGENCY_NORMALIZE = {
    "low": "Low",
    "medium": "Medium",
    "moderate": "Medium",
    "high": "High",
    "critical": "Critical",
    "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(value.lower().strip(), value)

# Re-parsing for base model nulls
def parse_output(raw):
    if not raw:
        return None
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

null_before = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Base model null entries BEFORE re-parsing: {null_before}/{len(base_predictions)}")

for p in base_predictions:
    if p["parsed_output"] is None:
        p["parsed_output"] = parse_output(p["raw_model_output"])

null_after = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Base model null entries AFTER re-parsing:  {null_after}/{len(base_predictions)}")
print(f"Recovered by re-parsing:                   {null_before - null_after}\n")

# Evaluation
def evaluate_prediction(pred):
    gt = pred["ground_truth"]
    out = pred["parsed_output"]

    if out is None:
        return 0.0

    score = 0

    # Department (0.3)
    if out.get("department") == gt.get("department"):
        score += 0.3

    # Symptoms (0.3)
    gt_symptoms = [s.lower() for s in gt.get("symptoms", [])]
    out_symptoms = [s.lower() for s in out.get("symptoms", [])]
    if gt_symptoms:
        matches = sum(1 for s in gt_symptoms if s in out_symptoms)
        if matches == len(gt_symptoms):
            score += 0.3
        else:
            score += (matches / len(gt_symptoms)) * 0.3

    # Sentiment (0.2)
    if out.get("sentiment") == gt.get("sentiment"):
        score += 0.2

    # Urgency level (0.2) — normalized
    if normalize_urgency(out.get("urgency_level")) == normalize_urgency(gt.get("urgency_level")):
        score += 0.2

    return round(score, 4)

# Scoring
def run_evaluation(preds, model_name):
    results = []
    for pred in preds:
        s = evaluate_prediction(pred)
        results.append({
            "id": pred["id"],
            "score": s,
            "valid_json": pred.get("valid_json", False)
        })
    scores = [r["score"] for r in results]
    avg = sum(scores) / len(scores)
    print(f"{model_name}")
    print(f"Total predictions: {len(scores)}")
    print(f"Average score:     {avg:.4f}")
    print(f"Min score:         {min(scores):.4f}")
    print(f"Max score:         {max(scores):.4f}\n")
    return results

base_results = run_evaluation(base_predictions, "Base Model")
ft_results   = run_evaluation(predictions, "Fine-tuned Model")

# Urgency direction analysis
URGENCY_ORDER = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}

def urgency_analysis(preds, model_name):
    lower, higher, wrong_other = 0, 0, 0
    for pred in preds:
        gt  = pred["ground_truth"]
        out = pred["parsed_output"]
        if out is None:
            continue
        gt_urgency  = normalize_urgency(gt.get("urgency_level"))
        out_urgency = normalize_urgency(out.get("urgency_level"))
        if gt_urgency == out_urgency:
            continue
        gt_rank  = URGENCY_ORDER.get(gt_urgency)
        out_rank = URGENCY_ORDER.get(out_urgency)
        if gt_rank is None or out_rank is None:
            wrong_other += 1
        elif out_rank < gt_rank:
            lower += 1
        elif out_rank > gt_rank:
            higher += 1

    total_wrong = lower + higher + wrong_other
    print(f"{model_name} Urgency Analysis")
    print(f"Total wrong urgency predictions: {total_wrong}")
    print(f"  Predicted LOWER than ground truth:  {lower}")
    print(f"  Predicted HIGHER than ground truth: {higher}")
    if wrong_other:
        print(f"  Unexpected/unrecognised values:     {wrong_other}")
    print()

urgency_analysis(base_predictions, "Base Model")
urgency_analysis(predictions, "Fine-tuned Model")

# Comparison
base_avg = sum(r["score"] for r in base_results) / len(base_results)
ft_avg   = sum(r["score"] for r in ft_results)   / len(ft_results)
print("Model Comparison")
print(f"Base model avg score:       {base_avg:.4f}")
print(f"Fine-tuned model avg score: {ft_avg:.4f}")
print(f"Improvement:                {ft_avg - base_avg:+.4f}")

Base model null entries BEFORE re-parsing: 1/760
Base model null entries AFTER re-parsing:  0/760
Recovered by re-parsing:                   1

Base Model
Total predictions: 760
Average score:     0.4246
Min score:         0.0000
Max score:         1.0000

Fine-tuned Model
Total predictions: 760
Average score:     0.7320
Min score:         0.0000
Max score:         1.0000

Base Model Urgency Analysis
Total wrong urgency predictions: 270
  Predicted LOWER than ground truth:  7
  Predicted HIGHER than ground truth: 263

Fine-tuned Model Urgency Analysis
Total wrong urgency predictions: 67
  Predicted LOWER than ground truth:  36
  Predicted HIGHER than ground truth: 31

Model Comparison
Base model avg score:       0.4246
Fine-tuned model avg score: 0.7320
Improvement:                +0.3074


In [34]:
import json
import re

# Load
with open('data/outputs/finetuned_model_predictions.json', 'r') as f:
    predictions = json.load(f)

with open('data/outputs/base_model_predictions.json', 'r') as f:
    base_predictions = json.load(f)


# Normalization
URGENCY_NORMALIZE = {
    "low": "Low", "medium": "Medium", "moderate": "Medium",
    "high": "High", "critical": "Critical", "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(value.lower().strip(), value)

#Re-parse base model nulls
def parse_output(raw):
    if not raw:
        return None
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

for p in base_predictions:
    if p["parsed_output"] is None:
        p["parsed_output"] = parse_output(p["raw_model_output"])


# 1. PER-DIMENSION ACCURACY TABLE

def per_dimension_accuracy(preds, model_name):
    n = len(preds)

    dept_correct      = 0
    sentiment_correct = 0
    urgency_correct   = 0

    # Symptoms: track full-match vs partial
    symptom_full      = 0
    symptom_partial   = 0
    symptom_none      = 0
    symptom_scores    = []

    null_outputs      = 0

    for pred in preds:
        gt  = pred["ground_truth"]
        out = pred["parsed_output"]

        if out is None:
            null_outputs += 1
            symptom_scores.append(0.0)
            continue

        # Department
        if out.get("department") == gt.get("department"):
            dept_correct += 1

        # Symptoms
        gt_s  = [s.lower() for s in gt.get("symptoms", [])]
        out_s = [s.lower() for s in out.get("symptoms", [])]
        if gt_s:
            ratio = sum(1 for s in gt_s if s in out_s) / len(gt_s)
            symptom_scores.append(ratio)
            if ratio == 1.0:
                symptom_full += 1
            elif ratio > 0:
                symptom_partial += 1
            else:
                symptom_none += 1
        else:
            symptom_scores.append(1.0)
            symptom_full += 1

        # Sentiment
        if out.get("sentiment") == gt.get("sentiment"):
            sentiment_correct += 1

        # Urgency (normalized)
        if normalize_urgency(out.get("urgency_level")) == normalize_urgency(gt.get("urgency_level")):
            urgency_correct += 1

    valid_n = n - null_outputs
    avg_symptom_score = sum(symptom_scores) / n if n else 0

    print(f"\n{'═'*52}")
    print(f"  Per-Dimension Accuracy — {model_name}")
    print(f"{'═'*52}")
    print(f"  Total predictions : {n}")
    print(f"  Null outputs      : {null_outputs}  ({100*null_outputs/n:.1f}%)")
    print(f"  Valid outputs     : {valid_n}")
    print(f"{'─'*52}")
    print(f"  {'Dimension':<22} {'Correct':>8} {'Accuracy':>10}")
    print(f"{'─'*52}")
    print(f"  {'Department':<22} {dept_correct:>8} {100*dept_correct/n:>9.1f}%")
    print(f"  {'Sentiment':<22} {sentiment_correct:>8} {100*sentiment_correct/n:>9.1f}%")
    print(f"  {'Urgency (normalized)':<22} {urgency_correct:>8} {100*urgency_correct/n:>9.1f}%")
    print(f"{'─'*52}")
    print(f"  Symptoms (avg match ratio) : {avg_symptom_score:.4f}  ({100*avg_symptom_score:.1f}%)")
    print(f"  Symptoms full match        : {symptom_full:>4}  ({100*symptom_full/n:.1f}%)")
    print(f"  Symptoms partial match     : {symptom_partial:>4}  ({100*symptom_partial/n:.1f}%)")
    print(f"  Symptoms no match          : {symptom_none:>4}  ({100*symptom_none/n:.1f}%)")
    print(f"{'═'*52}\n")

    return {
        "department":  dept_correct / n,
        "sentiment":   sentiment_correct / n,
        "urgency":     urgency_correct / n,
        "symptoms_avg": avg_symptom_score,
    }

base_dims = per_dimension_accuracy(base_predictions, "Base Model")
ft_dims   = per_dimension_accuracy(predictions,      "Fine-tuned Model")

# Side-by-side delta
print(f"\n{'═'*52}")
print(f"  Dimension-Level Improvement (FT − Base)")
print(f"{'═'*52}")
print(f"  {'Dimension':<25} {'Base':>8} {'FT':>8} {'Delta':>8}")
print(f"{'─'*52}")
for dim in ["department", "sentiment", "urgency", "symptoms_avg"]:
    b, f = base_dims[dim], ft_dims[dim]
    print(f"  {dim:<25} {100*b:>7.1f}% {100*f:>7.1f}% {100*(f-b):>+7.1f}%")
print(f"{'═'*52}\n")


════════════════════════════════════════════════════
  Per-Dimension Accuracy — Base Model
════════════════════════════════════════════════════
  Total predictions : 760
  Null outputs      : 0  (0.0%)
  Valid outputs     : 760
────────────────────────────────────────────────────
  Dimension               Correct   Accuracy
────────────────────────────────────────────────────
  Department                  360      47.4%
  Sentiment                   198      26.1%
  Urgency (normalized)        490      64.5%
────────────────────────────────────────────────────
  Symptoms (avg match ratio) : 0.6856  (68.6%)
  Symptoms full match        :  446  (58.7%)
  Symptoms partial match     :  146  (19.2%)
  Symptoms no match          :  168  (22.1%)
════════════════════════════════════════════════════


════════════════════════════════════════════════════
  Per-Dimension Accuracy — Fine-tuned Model
════════════════════════════════════════════════════
  Total predictions : 760
  Null outputs     

In [35]:
import json
import random

# Filter predictions scoring less than 0.8
low_scoring = [
    (pred, score)
    for pred, score in zip(base_predictions, base_scores)
    if score < 0.8
]

print(f"Total predictions scoring < 0.8: {len(low_scoring)}")
print(f"Sampling 5 for error analysis\n")

sample = random.sample(low_scoring, 5)

for pred, score in sample:
    gt  = pred["ground_truth"]
    out = pred["parsed_output"]

    print(f"{'='*60}")
    print(f"ID: {pred['id']}  |  Score: {score}")
    print(f"Question: {pred.get('question', 'N/A')[:200]}")
    print(f"\n  {'Field':<20} {'Ground Truth':<25} {'Predicted':<25}")
    print(f"  {'-'*70}")
    print(f"  {'Department':<20} {str(gt.get('department','')):<25} {str(out.get('department','') if out else 'NULL'):<25}")
    print(f"  {'Sentiment':<20} {str(gt.get('sentiment','')):<25} {str(out.get('sentiment','') if out else 'NULL'):<25}")
    print(f"  {'Urgency':<20} {str(gt.get('urgency_level','')):<25} {str(out.get('urgency_level','') if out else 'NULL'):<25}")
    print(f"\n  Symptoms GT  : {gt.get('symptoms', [])}")
    print(f"  Symptoms Out : {out.get('symptoms', []) if out else 'NULL'}")
    print()

Total predictions scoring < 0.8: 671
Sampling 5 for error analysis

ID: 719  |  Score: 0.5
Question: N/A

  Field                Ground Truth              Predicted                
  ----------------------------------------------------------------------
  Department           Cardiology                Cardiology               
  Sentiment            Concerned                 Concerned                
  Urgency              Medium                    High                     

  Symptoms GT  : ['palpitations', 'radiating pain to left scapula']
  Symptoms Out : ['intermittent palpitations', 'radiation to left scapula']

ID: 28  |  Score: 0.15
Question: N/A

  Field                Ground Truth              Predicted                
  ----------------------------------------------------------------------
  Department           Cardiology                Emergency                
  Sentiment            Worried                   Anxious                  
  Urgency              Medium          

## Improved fuzzy match for symptoms because of insights from error analysis

In [36]:
import json
import re

# Load predictions
with open('data/outputs/finetuned_model_predictions.json', 'r') as f:
    predictions = json.load(f)

with open('data/outputs/base_model_predictions.json', 'r') as f:
    base_predictions = json.load(f)


# Normalization
URGENCY_NORMALIZE = {
    "low": "Low",
    "medium": "Medium",
    "moderate": "Medium",
    "high": "High",
    "critical": "Critical",
    "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(value.lower().strip(), value)


# Re-parsing for base model nulls
def parse_output(raw):
    if not raw:
        return None
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

null_before = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Base model null entries BEFORE re-parsing: {null_before}/{len(base_predictions)}")

for p in base_predictions:
    if p["parsed_output"] is None:
        p["parsed_output"] = parse_output(p["raw_model_output"])

null_after = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Base model null entries AFTER re-parsing:  {null_after}/{len(base_predictions)}")
print(f"Recovered by re-parsing:                   {null_before - null_after}\n")


# Smarter Symptom Matching
def normalize_text(s):
    return re.sub(r'[^a-z0-9\s]', '', s.lower()).strip()

def symptom_match_score(gt_symptoms, pred_symptoms):
    if not gt_symptoms:
        return 1.0

    gt_norm = [normalize_text(s) for s in gt_symptoms]
    pred_norm = [normalize_text(s) for s in pred_symptoms]

    match_count = 0

    for gt in gt_norm:
        gt_tokens = set(gt.split())

        best_match = 0
        for pred in pred_norm:
            pred_tokens = set(pred.split())

            # Exact or substring match
            if gt in pred or pred in gt:
                best_match = 1
                break

            # Token overlap (Jaccard similarity)
            intersection = len(gt_tokens & pred_tokens)
            union = len(gt_tokens | pred_tokens)
            if union > 0:
                score = intersection / union
                best_match = max(best_match, score)

        match_count += best_match

    return match_count / len(gt_norm)


# Evaluation
def evaluate_prediction(pred):
    gt = pred["ground_truth"]
    out = pred["parsed_output"]

    if out is None:
        return 0.0

    score = 0

    # Department (0.3)
    if out.get("department") == gt.get("department"):
        score += 0.3

    # Symptoms (0.3) — improved matching
    gt_symptoms = gt.get("symptoms", [])
    out_symptoms = out.get("symptoms", [])
    symptom_score = symptom_match_score(gt_symptoms, out_symptoms)
    score += symptom_score * 0.3

    # Sentiment (0.2)
    if out.get("sentiment") == gt.get("sentiment"):
        score += 0.2

    # Urgency (0.2)
    if normalize_urgency(out.get("urgency_level")) == normalize_urgency(gt.get("urgency_level")):
        score += 0.2

    return round(score, 4)


# Run Evaluation
def run_evaluation(preds, model_name):
    results = []

    for pred in preds:
        s = evaluate_prediction(pred)
        results.append({
            "id": pred["id"],
            "score": s,
            "valid_json": pred.get("valid_json", False)
        })

    scores = [r["score"] for r in results]
    avg = sum(scores) / len(scores)

    print(f"{model_name}")
    print(f"Total predictions: {len(scores)}")
    print(f"Average score:     {avg:.4f}")
    print(f"Min score:         {min(scores):.4f}")
    print(f"Max score:         {max(scores):.4f}\n")

    return results


base_results = run_evaluation(base_predictions, "Base Model")
ft_results   = run_evaluation(predictions, "Fine-tuned Model")


#Urgency Direction Analysis
URGENCY_ORDER = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}

def urgency_analysis(preds, model_name):
    lower, higher, wrong_other = 0, 0, 0

    for pred in preds:
        gt  = pred["ground_truth"]
        out = pred["parsed_output"]

        if out is None:
            continue

        gt_urgency  = normalize_urgency(gt.get("urgency_level"))
        out_urgency = normalize_urgency(out.get("urgency_level"))

        if gt_urgency == out_urgency:
            continue

        gt_rank  = URGENCY_ORDER.get(gt_urgency)
        out_rank = URGENCY_ORDER.get(out_urgency)

        if gt_rank is None or out_rank is None:
            wrong_other += 1
        elif out_rank < gt_rank:
            lower += 1
        elif out_rank > gt_rank:
            higher += 1

    total_wrong = lower + higher + wrong_other

    print(f"{model_name} Urgency Analysis")
    print(f"Total wrong urgency predictions: {total_wrong}")
    print(f"  Predicted LOWER than ground truth:  {lower}")
    print(f"  Predicted HIGHER than ground truth: {higher}")
    if wrong_other:
        print(f"  Unexpected/unrecognised values:     {wrong_other}")
    print()


urgency_analysis(base_predictions, "Base Model")
urgency_analysis(predictions, "Fine-tuned Model")


# Final Comparison
base_avg = sum(r["score"] for r in base_results) / len(base_results)
ft_avg   = sum(r["score"] for r in ft_results)   / len(ft_results)

print("Model Comparison")
print(f"Base model avg score:       {base_avg:.4f}")
print(f"Fine-tuned model avg score: {ft_avg:.4f}")
print(f"Improvement:                {ft_avg - base_avg:+.4f}")

Base model null entries BEFORE re-parsing: 1/760
Base model null entries AFTER re-parsing:  0/760
Recovered by re-parsing:                   1

Base Model
Total predictions: 760
Average score:     0.5849
Min score:         0.0000
Max score:         1.0000

Fine-tuned Model
Total predictions: 760
Average score:     0.8609
Min score:         0.0000
Max score:         1.0000

Base Model Urgency Analysis
Total wrong urgency predictions: 270
  Predicted LOWER than ground truth:  7
  Predicted HIGHER than ground truth: 263

Fine-tuned Model Urgency Analysis
Total wrong urgency predictions: 67
  Predicted LOWER than ground truth:  36
  Predicted HIGHER than ground truth: 31

Model Comparison
Base model avg score:       0.5849
Fine-tuned model avg score: 0.8609
Improvement:                +0.2760


## Fuzzy Match change-effect on symptoms match

In [37]:
import json
import re

# Load
with open('data/outputs/finetuned_model_predictions.json', 'r') as f:
    predictions = json.load(f)

with open('data/outputs/base_model_predictions.json', 'r') as f:
    base_predictions = json.load(f)

# Normalization
URGENCY_NORMALIZE = {
    "low": "Low", "medium": "Medium", "moderate": "Medium",
    "high": "High", "critical": "Critical", "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(value.lower().strip(), value)

# Re-parse base model nulls
def parse_output(raw):
    if not raw:
        return None
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

for p in base_predictions:
    if p["parsed_output"] is None:
        p["parsed_output"] = parse_output(p["raw_model_output"])

# Fuzzy symptom matching
def symptom_match_ratio(gt_symptoms, out_symptoms):
    if not gt_symptoms:
        return 1.0

    def tokenize(s):
        return set(s.lower().replace("-", " ").split())

    total_ratio = 0.0
    for gt_s in gt_symptoms:
        gt_tokens = tokenize(gt_s)
        best = 0.0
        for out_s in out_symptoms:
            out_tokens = tokenize(out_s)
            if not gt_tokens:
                continue
            overlap = len(gt_tokens & out_tokens) / len(gt_tokens)
            best = max(best, overlap)
        total_ratio += best

    return total_ratio / len(gt_symptoms)


# 1. PER-DIMENSION ACCURACY TABLE

def per_dimension_accuracy(preds, model_name):
    n = len(preds)

    dept_correct      = 0
    sentiment_correct = 0
    urgency_correct   = 0

    symptom_full      = 0
    symptom_partial   = 0
    symptom_none      = 0
    symptom_scores    = []

    null_outputs      = 0

    for pred in preds:
        gt  = pred["ground_truth"]
        out = pred["parsed_output"]

        if out is None:
            null_outputs += 1
            symptom_scores.append(0.0)
            continue

        # Department
        if out.get("department") == gt.get("department"):
            dept_correct += 1

        # Symptoms — fuzzy token overlap
        gt_s  = gt.get("symptoms", [])
        out_s = out.get("symptoms", [])
        ratio = symptom_match_ratio(gt_s, out_s)
        symptom_scores.append(ratio)
        if ratio == 1.0:
            symptom_full += 1
        elif ratio > 0:
            symptom_partial += 1
        else:
            symptom_none += 1

        # Sentiment
        if out.get("sentiment") == gt.get("sentiment"):
            sentiment_correct += 1

        # Urgency (normalized)
        if normalize_urgency(out.get("urgency_level")) == normalize_urgency(gt.get("urgency_level")):
            urgency_correct += 1

    valid_n = n - null_outputs
    avg_symptom_score = sum(symptom_scores) / n if n else 0

    print(f"\n{'═'*52}")
    print(f"  Per-Dimension Accuracy — {model_name}")
    print(f"{'═'*52}")
    print(f"  Total predictions : {n}")
    print(f"  Null outputs      : {null_outputs}  ({100*null_outputs/n:.1f}%)")
    print(f"  Valid outputs     : {valid_n}")
    print(f"{'─'*52}")
    print(f"  {'Dimension':<22} {'Correct':>8} {'Accuracy':>10}")
    print(f"{'─'*52}")
    print(f"  {'Department':<22} {dept_correct:>8} {100*dept_correct/n:>9.1f}%")
    print(f"  {'Sentiment':<22} {sentiment_correct:>8} {100*sentiment_correct/n:>9.1f}%")
    print(f"  {'Urgency (normalized)':<22} {urgency_correct:>8} {100*urgency_correct/n:>9.1f}%")
    print(f"{'─'*52}")
    print(f"  Symptoms (avg match ratio) : {avg_symptom_score:.4f}  ({100*avg_symptom_score:.1f}%)")
    print(f"  Symptoms full match        : {symptom_full:>4}  ({100*symptom_full/n:.1f}%)")
    print(f"  Symptoms partial match     : {symptom_partial:>4}  ({100*symptom_partial/n:.1f}%)")
    print(f"  Symptoms no match          : {symptom_none:>4}  ({100*symptom_none/n:.1f}%)")
    print(f"{'═'*52}\n")

    return {
        "department":   dept_correct / n,
        "sentiment":    sentiment_correct / n,
        "urgency":      urgency_correct / n,
        "symptoms_avg": avg_symptom_score,
    }

base_dims = per_dimension_accuracy(base_predictions, "Base Model")
ft_dims   = per_dimension_accuracy(predictions,      "Fine-tuned Model")

# Side-by-side delta
print(f"\n{'═'*52}")
print(f"  Dimension-Level Improvement (FT − Base)")
print(f"{'═'*52}")
print(f"  {'Dimension':<25} {'Base':>8} {'FT':>8} {'Delta':>8}")
print(f"{'─'*52}")
for dim in ["department", "sentiment", "urgency", "symptoms_avg"]:
    b, f = base_dims[dim], ft_dims[dim]
    print(f"  {dim:<25} {100*b:>7.1f}% {100*f:>7.1f}% {100*(f-b):>+7.1f}%")
print(f"{'═'*52}\n")


════════════════════════════════════════════════════
  Per-Dimension Accuracy — Base Model
════════════════════════════════════════════════════
  Total predictions : 760
  Null outputs      : 0  (0.0%)
  Valid outputs     : 760
────────────────────────────────────────────────────
  Dimension               Correct   Accuracy
────────────────────────────────────────────────────
  Department                  360      47.4%
  Sentiment                   198      26.1%
  Urgency (normalized)        490      64.5%
────────────────────────────────────────────────────
  Symptoms (avg match ratio) : 0.8788  (87.9%)
  Symptoms full match        :  560  (73.7%)
  Symptoms partial match     :  177  (23.3%)
  Symptoms no match          :   23  (3.0%)
════════════════════════════════════════════════════


════════════════════════════════════════════════════
  Per-Dimension Accuracy — Fine-tuned Model
════════════════════════════════════════════════════
  Total predictions : 760
  Null outputs      